In [ ]:
import rasterio, rasterio.mask
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import rasterio.features as features

from shapely.geometry import Point
from scipy.spatial import distance_matrix, distance
from rasterio.transform import Affine

## reading and joining aerophotos data

In [ ]:
# reading and editing data on aerophotos for year 2014

df2014 = pd.read_csv('Aerofotografije_2014.csv', encoding='latin-1', delimiter=';')

df2014['e'] = df2014['e'].str.replace(',', '.').astype('float64')
df2014['n'] = df2014['n'].str.replace(',', '.').astype('float64')
df2014['H'] = df2014['H'].str.replace(',', '.').astype('float64')
df2014['H_2010'] = df2014['H_2010'].str.replace(',', '.').astype('float64')
df2014['Omega'] = df2014['Omega'].str.replace(',', '.').astype('float64')
df2014['Fi'] = df2014['Fi'].str.replace(',', '.').astype('float64')
df2014['Kapa'] = df2014['Kapa'].str.replace(',', '.').astype('float64')

df2014['geometry'] = [Point(coords[0], coords[1]) for coords in zip(df2014['e'], df2014['n'])]

In [ ]:
# reading and editing data on aerophotos for year 2015

df2015 = pd.read_csv('Aerofotografije_2015.csv', encoding='latin-1', delimiter=';')

df2015['e'] = df2015['e'].str.replace(',', '.').astype('float64')
df2015['n'] = df2015['n'].str.replace(',', '.').astype('float64')
df2015['H'] = df2015['H'].str.replace(',', '.').astype('float64')
df2015['H_2010'] = df2015['H_2010'].str.replace(',', '.').astype('float64')
df2015['Omega'] = df2015['Omega'].str.replace(',', '.').astype('float64')
df2015['Fi'] = df2015['Fi'].str.replace(',', '.').astype('float64')
df2015['Kapa'] = df2015['Kapa'].str.replace(',', '.').astype('float64')

df2015['geometry'] = [Point(coords[0], coords[1]) for coords in zip(df2015['e'], df2015['n'])]

In [ ]:
# merging data of aerophotos for both years

gdf = gpd.GeoDataFrame(pd.concat([df2014, df2015]), geometry='geometry')

# gdf.to_file('aerophotos_merged.shp')

gdf = gdf.set_index('ID')

In [ ]:
# reading seamlines data and joining them with aerophotos data

sivne_linije = gpd.read_file('sivne_linije.shp')
sivne_linije = sivne_linije.set_index('AF_sifra')

join = sivne_linije.join(gdf, lsuffix='_caller', rsuffix='_other')
join = join.set_geometry('geometry_caller')

In [ ]:
# reading aeras of interest data

obmocja = gpd.read_file('skupaj.shp')

## angle calculation for the entire area of interest

In [ ]:
# function to calculate distance from pixel centre to projection centre

def calculate_euclidean(out_shape, x_plane, y_plane, transform):
    height, width = out_shape
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs_ys = rasterio.transform.xy(transform, rows, cols)
    
    coords = np.array(xs_ys).transpose(1, 2, 0)
    coord_list = coords.reshape((-1, 2))
    
    euclidean = distance_matrix(coord_list, [[x_plane, y_plane]], p=2)
    euclidean_twodimensional = euclidean.reshape((coords.shape[0], coords.shape[1]))
    
    return euclidean_twodimensional

In [ ]:
# function to calculate height above ground

def height_from_gdf(shape, height, out_shape, dtm, transform):
    # array pove, kje nas višina nad terenom zanima (kje je maska šivne linije)
    array = features.rasterize(zip([shape], [height]), out_shape=out_shape, transform=transform)
    
    # izračun višine letala nad terenom
    height_above_ground = array[array != 0] - dtm[array != 0]
    array[array != 0] = height_above_ground
    
    return array

In [ ]:
# function to merge heights and angles

def calculate_euclidean_photo(sivne_linije, out_shape, dtm, transform):
    shape = sivne_linije.geometry_caller
    height = sivne_linije.H_2010
    
    x_coord = sivne_linije.e
    y_coord = sivne_linije.n
    
    height = height_from_gdf(shape, height, out_shape, dtm, transform)
    
    euclidean_photo = calculate_euclidean(out_shape, x_coord, y_coord, transform)
    
    euclidean_photo[height == 0.0] = 0
    
    return euclidean_photo, height

In [ ]:
# function to calculate displacement

def calculate_displacement(euclidean_photo, chm, plane_height):
    displacement = ((chm * euclidean_photo) / (plane_height - chm))

    return displacement

In [ ]:
# function to convert radians to degrees

def rad_to_deg(radian_angles):
    degree_angles = radian_angles * (180 / np.pi)
    return degree_angles

In [ ]:
# function to calculate angles and displacement for the entire area of interest

def area_angle_displacement_calculations(name, area, chm_vrt, dtm_vrt):
    minx, miny, maxx, maxy = area.bounds
    
    cols = (maxx - minx) / 0.5
    rows = (maxy - miny) / 0.5
    out_shape = (int(rows), int(cols))
    
    transform = Affine(0.5, 0.0, minx, 0.0, -0.5, maxy)
    
    area_gdf = join[join['name']==name]
    
    euclidean_together = np.zeros(out_shape)
    height_together = np.zeros(out_shape)
    
    with rasterio.open(chm_vrt) as vrt:
        chm, out_transform = rasterio.mask.mask(vrt, [area], crop=True)
        crs = vrt.crs
    
    with rasterio.open(dtm_vrt) as vrt:
        dtm, _ = rasterio.mask.mask(vrt, [area], crop=True)

    for sl in area_gdf.iterrows():
        euclidean, height = calculate_euclidean_photo(sl[1], out_shape, dtm[0], transform)
        
        euclidean_together = euclidean_together + euclidean
        height_together = height_together + height
        
    displacement = calculate_displacement(euclidean_together, chm[0], height_together)
    
    angles = np.arctan(euclidean_together / (height_together - chm[0]))
    angles = rad_to_deg(angles)
    
    return angles, displacement, out_transform, crs

In [ ]:
# running displacement calculation

for obmocje in obmocja.iterrows():
    name = obmocje[1][2]
    area = obmocje[1][4]
    
    chm_vrt = r'i:\adam\doktorat\data\vrt\als_' + name + '.vrt'
    dtm_vrt = r'i:\adam\doktorat\data\vrt\dmv_' + name + '.vrt'
    
    angles, displacement, out_transform, crs = area_angle_displacement_calculations(name, area, chm_vrt, dtm_vrt)
    
    angles_tif = r'i:\adam\doktorat\data\calculate_displacement\angles_' + name + '.tif'
    displacement_tif = r'i:\adam\doktorat\data\calculate_displacement\displacement_' + name + '.tif'
    
    with rasterio.open(angles_tif, mode='w', driver='GTiff', height=angles.shape[0], width=angles.shape[1],
                       count=1, dtype=angles.dtype, crs=crs, transform=out_transform) as dst:
        dst.write(angles, 1)
    
    with rasterio.open(displacement_tif, mode='w', driver='GTiff', height=displacement.shape[0], width=displacement.shape[1],
                       count=1, dtype=displacement.dtype, crs=crs, transform=out_transform) as dst:
        dst.write(displacement, 1)
